# CogLog seven-segment Swift-YOLO retrain (Colab free)

- Trainer: [Seeed-Studio/ModelAssistant](https://github.com/Seeed-Studio/ModelAssistant) (**Apache License 2.0**)
- Dataset: [LibreYOLO/digits-t2eg6](https://huggingface.co/datasets/LibreYOLO/digits-t2eg6) (CC BY 4.0)
- Target contract: input **192**, **10** digit classes (0–9), **TFLite FLOAT32** (NHWC, score÷100)

Upload this notebook to Google Colab, set **Runtime → GPU**, then run all cells.
Details: `tool/seven_segment/colab_retrain.md` in the CogLog repo.

Do **not** use SSCMA `tools/export.py` for the Android asset — it emits INT8 only.

## 0. GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → GPU (T4)"

## 1. Mount Drive (checkpoints live here)

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/coglog_seven_segment")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
# Training work_dir on Drive so free-Colab disconnects do not wipe checkpoints.
WORK_DIR = DRIVE_ROOT / "work_dirs" / "coglog_digits192"
WORK_DIR.mkdir(parents=True, exist_ok=True)
EXPORTS = DRIVE_ROOT / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)
print(WORK_DIR)

## 2. Fetch CogLog helper scripts

Uses the public CogLog tree. Change `COGLOG_REF` if you need a PR branch (e.g. `cursor/colab-seven-seg-retrain-0fba`).

In [ ]:
import os
import subprocess
from pathlib import Path

COGLOG_REF = os.environ.get("COGLOG_REF", "main")
WORK = Path("/content/coglog_seven_segment")
WORK.mkdir(parents=True, exist_ok=True)
%cd /content
!rm -rf /content/CogLog_sparse
!git clone --depth 1 --filter=blob:none --sparse https://github.com/madahataraku/CogLog.git CogLog_sparse
%cd /content/CogLog_sparse
!git sparse-checkout set tool/seven_segment
subprocess.check_call(["git", "fetch", "--depth", "1", "origin", COGLOG_REF])
subprocess.check_call(["git", "checkout", "FETCH_HEAD"])
TOOLS = Path("/content/CogLog_sparse/tool/seven_segment")
assert (TOOLS / "prepare_digits_t2eg6.py").is_file()
assert (TOOLS / "export_tflite_float32.py").is_file()
print("helpers", TOOLS, "ref", COGLOG_REF)

## 3. Install ModelAssistant and patch deepen_factor

Same patch as `convert_coreml.py` so Colab `.pth` loads on macOS without `state_dict mismatch`.

In [ ]:
import sys
from pathlib import Path

%cd /content
!rm -rf ModelAssistant
!git clone --depth 1 https://github.com/Seeed-Studio/ModelAssistant.git
%cd /content/ModelAssistant
!python -m pip -q install -U pip wheel
!python -m pip -q install -r requirements.txt
!python -m pip -q install pillow huggingface_hub onnx onnxsim onnx2tf tensorflow

sys.path.insert(0, str(TOOLS))
from sscma_seven_seg_export_common import ensure_yolov5_csp_deepen_factor

SSCMA_ROOT = Path("/content/ModelAssistant")
ensure_yolov5_csp_deepen_factor(SSCMA_ROOT)
!python tools/train.py --help | head -n 5

## 4. Download digits-t2eg6 and convert to COCO

In [ ]:
DATA = WORK / "datasets/digits_t2eg6"
!python "{TOOLS}/prepare_digits_t2eg6.py" --work-dir "{DATA}" --symlink-images
COCO = DATA / "coco"
assert (COCO / "train" / "_annotations.coco.json").is_file()
print("coco", COCO)

## 5. Train Swift-YOLO tiny @ 192

`WORK_DIR` is on Drive. Free Colab: start with `EPOCHS=50`. Re-run with `--resume` if the session drops.

In [ ]:
import os
import subprocess
from pathlib import Path

%cd /content/ModelAssistant
EPOCHS = int(os.environ.get("EPOCHS", "50"))
BATCH = int(os.environ.get("BATCH", "16"))
RESUME = os.environ.get("RESUME", "0") == "1"
cmd = [
    "python", "tools/train.py",
    "configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py",
    "--work-dir", str(WORK_DIR),
    "--cfg-options",
    "num_classes=10",
    "width=192",
    "height=192",
    f"batch={BATCH}",
    f"epochs={EPOCHS}",
    f"data_root={COCO}/",
    "train_ann=train/_annotations.coco.json",
    "train_data=train/",
    "val_ann=valid/_annotations.coco.json",
    "val_data=valid/",
]
# ModelAssistant comments out cfg.resume = args.resume; set via cfg-options.
if RESUME:
    cmd.append("resume=True")
print(" ".join(cmd))
subprocess.check_call(cmd)

## 6. Export TFLite FLOAT32 (CogLog contract)

Uses `export_tflite_float32.py` (NHWC + score÷100). Not SSCMA INT8 export.

In [ ]:
from pathlib import Path

ckpt = WORK_DIR / "last_checkpoint"
assert ckpt.is_file(), ckpt
weights = Path(ckpt.read_text().strip())
if not weights.is_file():
    weights = WORK_DIR / weights.name
assert weights.is_file(), weights
print("weights", weights)

import subprocess

import numpy as np
import tensorflow as tf

TFLITE_OUT = EXPORTS / "digits_t2eg6_swift_yolo_192_float32.tflite"
subprocess.check_call([
    "python", str(TOOLS / "export_tflite_float32.py"),
    "--weights", str(weights),
    "--sscma-root", str(SSCMA_ROOT),
    "--output", str(TFLITE_OUT),
])

interpreter = tf.lite.Interpreter(model_path=str(TFLITE_OUT))
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
assert inp["dtype"] == np.float32 and out["dtype"] == np.float32, (inp["dtype"], out["dtype"])
assert tuple(int(x) for x in inp["shape"]) == (1, 192, 192, 3), list(inp["shape"])
print("verified float32 NHWC", list(inp["shape"]), list(out["shape"]))

## 7. Copy `.pth` + NOTICE to Drive exports

In [ ]:
import shutil
from pathlib import Path

shutil.copy2(weights, EXPORTS / weights.name)
notice_src = DATA / "NOTICE.md"
if notice_src.is_file():
    shutil.copy2(notice_src, EXPORTS / "DATASET_NOTICE.md")
print("exports", list(EXPORTS.iterdir()))
print("Next on macOS: convert_coreml.md path B with this .pth")